In [13]:
import duckdb

PARQUET_PATH = "../data/processed/edinburghcstr_ami/combined_features_with_transcripts.parquet"

# 1. Gerçek Oracle Dağılımı (Tüm Dataset İçin)
# Eşit WER durumlarında önceliği sırayla Hubert -> Whisper -> W2V2'ye veriyoruz.
oracle_query = f"""
SELECT 
    oracle_model, 
    COUNT(*) as count, 
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM (
    SELECT 
        CASE 
            WHEN hubert_wer <= whisper_wer AND hubert_wer <= w2v2_wer THEN 'hubert'
            WHEN whisper_wer <= hubert_wer AND whisper_wer <= w2v2_wer THEN 'whisper'
            ELSE 'w2v2' 
        END AS oracle_model
    FROM '{PARQUET_PATH}'
) 
GROUP BY oracle_model
ORDER BY percentage DESC
"""

# 2. Modeller Arası Hata Korelasyonu (Pearson)
# Hangi modellerin aynı akustik zorluklarda çuvalladığını gösterir.
corr_query = f"""
SELECT 
    ROUND(corr(hubert_wer, w2v2_wer), 4) as hubert_w2v2_corr,
    ROUND(corr(hubert_wer, whisper_wer), 4) as hubert_whisper_corr,
    ROUND(corr(w2v2_wer, whisper_wer), 4) as w2v2_whisper_corr
FROM '{PARQUET_PATH}'
"""

print("=== ORACLE FREQUENCY (TÜM DATASET) ===")
oracle_df = duckdb.query(oracle_query).df()
print(oracle_df.to_string(index=False))

print("\n=== ERROR CORRELATION (MODEL TAMAMLAYICILIĞI) ===")
corr_df = duckdb.query(corr_query).df()
print(corr_df.to_string(index=False))

=== ORACLE FREQUENCY (TÜM DATASET) ===
oracle_model  count  percentage
     whisper   5595       44.36
      hubert   4074       32.30
        w2v2   2943       23.33

=== ERROR CORRELATION (MODEL TAMAMLAYICILIĞI) ===
 hubert_w2v2_corr  hubert_whisper_corr  w2v2_whisper_corr
           0.3475               0.0466             0.1015


### 1. Oracle Frekansı: "Nöral Ağın Kör Noktası"
Veri setinin bütününe baktığımızda **gerçek Oracle'da HuBERT %32.3 oranında en iyi modelmiş!**
Yani her 3 ses klibinden 1'inde HuBERT, diğer iki modele göre daha iyi (veya en azından eşit) transkripsiyon üretiyor. 

Fakat hatırlarsan senin loglarında `test/freq_hubert` değeri %1 (MLP) ve %3 (Transformer) civarındaydı. Bu durum literatürde **Mode Collapse (Mod Çökmesi)** veya **Shadowing (Gölgelenme)** olarak adlandırılır. Senin router (seçici) ağın, %32'lik bu potansiyeli bulmayı öğrenememiş ve güvenli liman olan diğer iki modele kaçmış. Peki ama neden? Cevap korelasyon tablosunda.

### 2. Hata Korelasyonu: "HuBERT Neden Gölgeleniyor?"
Korelasyon tablosu, modellerin hangi örneklerde çuvalladığının benzerliğini gösteriyor:
* **HuBERT & W2V2 (0.3475):** Aralarındaki hata korelasyonu en yüksek olan ikili. İkisi de benzer kontrastif öğrenme/self-supervised mimarilere sahip oldukları için aynı akustik zorluklarda (örneğin arka plan gürültüsü, aksan vs.) benzer şekilde hata yapıyorlar.
* **HuBERT & Whisper (0.0466):** Korelasyon neredeyse sıfır! Yani tamamen bağımsız hata yapıyorlar. Biri batırırken diğeri mükemmel çıkarabiliyor.

**Sentez: Ağ Neden HuBERT'i Seçmiyor?**
Nöral ağın eğitim sırasındaki mantığı (Loss'u minimize etme çabası) şu şekilde çalışıyor:
1. Bir klip geliyor, ağ akustik özelliklere bakıyor.
2. Bu özellikler "self-supervised (HuBERT/W2V2) mimarilerin iyi çözdüğü" türden bir uzaydaysa, ağ Wav2Vec2'yi seçiyor. Çünkü ortalama başarıda (Single Baseline) Wav2Vec2 (WER 0.39), HuBERT'ten (WER 0.63) çok daha güçlü. İkisi de aynı yerlerde hata yaptığı için, ağ *"Madem aynı mantıkla çalışıyorlar, genel olarak daha başarılı olan Wav2Vec2'ye oynayayım"* diyor. 
3. Eğer klip self-supervised modellerin zorlandığı bir tipteyse, ağ bambaşka bir uzaydan gelen (korelasyon: 0.04 - 0.10) Whisper'ı seçerek riski dağıtıyor.
4. Sonuç olarak HuBERT, Wav2Vec2'nin gölgesinde kalıyor (Overshadowed).

### Modelini Geliştirmek İçin Ne Yapabilirsin?
Artık problemin veri ve mimari etkileşiminden kaynaklandığını biliyoruz. HuBERT'in o %32'lik gizli potansiyelini ağa öğretebilmek için şu deneyleri yapabilirsin:

1. **Ağırlıklı Loss (Class-Balanced CE):** Seçici ağın (router) HuBERT'i seçmesi gerektiği halde seçmediği durumlarda uygulanan cezayı (Loss'u) artırabilirsin. `hard_ce` loss hesaplarken, sınıf frekanslarının tersiyle (veya yumuşatılmış tersiyle) ağırlıklandırma yapabilirsin.
2. **Soft Label Sıcaklığı (Temperature):** `soft-ce-temperature 0.1` kullanıyorsun. Bu çok "hard" (keskin) bir dağılım yaratır. Bunu `1.0` veya `2.0` gibi daha yüksek bir değere çekerek, modellerin birbirine yakın performans gösterdiği durumlarda ağın daha yumuşak güncellemeler almasını sağlayabilirsin. 0.1 sıcaklık, "kazanan her şeyi alır" mantığına yol açıp HuBERT'i sıfırlamış olabilir.
3. **Girdi Özelliklerinin (Feature) Normalizasyonu:** Modellerin (Whisper, HuBERT, W2V2) ürettiği embedding'lerin büyüklükleri (L2 normları) farklı olabilir. Örneğin Whisper embedding'leri daha yüksek varyansa sahipse, MLP veya Transformer onlara daha fazla odaklanıyordur. Modellerden gelen feature'ları (veya cross-attention öncesi) LayerNorm'dan geçirdiğine emin ol.

Bu analizle makale için harika bir "Discussion/Analysis" bölümü malzemesi çıkarmış olduk!